# 1. Data Ingestion - NYC TLC Trip Record Data

This notebook handles the ingestion of NYC Taxi and Limousine Commission (TLC) trip record data.

**Dataset:** NYC TLC Trip Record Data (Yellow Taxi)
- **Source:** https://registry.opendata.aws/nyc-tlc-trip-records-pds/
- **Size:** Multiple GB (we'll use 2019-2020 data)
- **Features:** 18+ columns including pickup/dropoff times, locations, fares, distances, etc.
- **Problem Type:** Regression (predicting trip duration or fare amount)

In [1]:
# Import required libraries
import os
import yaml
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load Spark configuration
with open('../config/spark_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

spark_config = config['spark']
print("Configuration loaded:", spark_config)

Configuration loaded: {'app_name': 'ML_Pipeline', 'master': 'local[*]', 'driver_memory': '2g', 'executor_memory': '2g', 'executor_cores': 4, 'shuffle_partitions': 200, 'sql': {'shuffle_partitions': 200, 'adaptive_execution': True}, 'speculation': False, 'scheduler_mode': 'FIFO'}


In [ ]:
# Initialize Spark Session with optimized configuration
spark = SparkSession.builder \
    .appName(spark_config['app_name']) \
    .master(spark_config['master']) \
    .config("spark.driver.memory", spark_config['driver_memory']) \
    .config("spark.executor.memory", spark_config['executor_memory']) \
    .config("spark.executor.cores", spark_config['executor_cores']) \
    .config("spark.sql.shuffle.partitions", spark_config['shuffle_partitions']) \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## Data Schema Definition

Define the schema for NYC TLC Yellow Taxi data to ensure proper data types and optimize loading.

In [ ]:
# Define schema for NYC TLC data
nyc_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True)
])

print("Schema defined successfully!")

## Data Loading

Load NYC TLC data from Parquet files. The data is available on AWS S3.

In [ ]:
# Option 1: Load from AWS S3 (public dataset)
# Note: This requires internet connection and may take time
# Uncomment to use:
# base_url = "s3a://nyc-tlc/trip data/yellow_tripdata_2019-*.parquet"

# Option 2: Load from local files (recommended for assignment)
# Download data first from: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Place parquet files in ../data/raw/

data_path = "../data/raw/yellow_tripdata_*.parquet"

# Load data with schema
df = spark.read \
    .schema(nyc_schema) \
    .parquet(data_path)

print(f"Data loaded successfully!")
print(f"Total records: {df.count():,}")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

In [ ]:
# Display schema
df.printSchema()

In [ ]:
# Show sample data
df.show(10, truncate=False)

## Initial Data Quality Checks

In [ ]:
# Check for null values
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts_pd = null_counts.toPandas().T
null_counts_pd.columns = ['null_count']
null_counts_pd['null_percentage'] = (null_counts_pd['null_count'] / df.count()) * 100
print("\nNull Value Analysis:")
print(null_counts_pd[null_counts_pd['null_count'] > 0])

In [ ]:
# Basic statistics
df.describe().show()

In [ ]:
# Check data size
size_bytes = df.rdd.map(lambda x: len(str(x))).sum()
size_gb = size_bytes / (1024**3)
print(f"Approximate data size: {size_gb:.2f} GB")

## Data Validation and Filtering

In [ ]:
# Filter out invalid records
df_clean = df.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0) &
    (col("passenger_count") <= 6) &
    (col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"))
)

print(f"Records before filtering: {df.count():,}")
print(f"Records after filtering: {df_clean.count():,}")
print(f"Records removed: {df.count() - df_clean.count():,}")

## Save Cleaned Data

In [ ]:
# Repartition for optimal storage
df_clean = df_clean.repartition(50)

# Save as Parquet with partitioning
output_path = "../data/processed/nyc_tlc_clean"
df_clean.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Cleaned data saved to: {output_path}")

In [ ]:
# Save sample for quick analysis
sample_df = df_clean.sample(fraction=0.01, seed=42)
sample_df.write \
    .mode("overwrite") \
    .parquet("../data/samples/nyc_tlc_sample")

print(f"Sample data saved with {sample_df.count():,} records")

## Summary Statistics for Report

In [ ]:
# Generate summary statistics
summary_stats = {
    'total_records': df_clean.count(),
    'total_columns': len(df_clean.columns),
    'date_range': {
        'min': df_clean.agg(min('tpep_pickup_datetime')).collect()[0][0],
        'max': df_clean.agg(max('tpep_pickup_datetime')).collect()[0][0]
    },
    'avg_trip_distance': df_clean.agg(avg('trip_distance')).collect()[0][0],
    'avg_fare_amount': df_clean.agg(avg('fare_amount')).collect()[0][0]
}

print("\nDataset Summary:")
for key, value in summary_stats.items():
    print(f"{key}: {value}")

In [ ]:
# Stop Spark session
# spark.stop()
print("Data ingestion complete!")